# Marketing Agent — SFT → DPO → Eval (SageMaker Studio Lab)

Full training + evaluation on **free** SageMaker Studio Lab (no credits, no card).

**Before you start:**
1. In the launcher, set the compute to **GPU** (top-right runtime selector), then *Start runtime*.
   Studio Lab gives a **T4** with a **4-hour** GPU session limit — plenty for a 3B QLoRA run.
2. Your project files **persist** across sessions (~15GB), so adapters survive a restart.
3. Run Phases 1-3 first (locally or in `colab_data_pipeline.ipynb`) and **push**
   `data/preferences/preferences.jsonl` — `git clone` below brings it in.
4. Have your `GEMINI_API_KEY` ready for the judging step.


## 0. Confirm the GPU is attached

If this fails, you're on the CPU runtime — switch to GPU and restart.

In [ ]:
!nvidia-smi

## 1. Clone the repo + install training deps

Studio Lab persists the project dir, so subsequent sessions can skip the clone.

In [ ]:
REPO_URL = 'https://github.com/SalahnAI/Preference-Optimization-for-Marketing-Agents.git'
import os
if not os.path.isdir('Preference-Optimization-for-Marketing-Agents'):
    !git clone $REPO_URL
%cd Preference-Optimization-for-Marketing-Agents

In [ ]:
# Installs into the active Studio Lab conda env. ~2-3 min the first time.
!pip install -q -r training/requirements-train.txt

## 2. Sanity-check the preference dataset

In [ ]:
import json, pathlib
p = pathlib.Path('data/preferences/preferences.jsonl')
assert p.exists(), 'preferences.jsonl missing — run Phases 1-3 and push it first.'
rows = [json.loads(l) for l in p.open()]
print(f'{len(rows)} preference pairs')
print('\n--- example chosen ---\n', rows[0]['chosen'][:400])

## 3. Pick the base model

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'  # or meta-llama/Llama-3.2-3B-Instruct (gated)

## 4. Phase 5 — SFT (LoRA on the *chosen* responses)

~10-20 min on the T4. Keep an eye on the 4h GPU clock.

In [ ]:
!python training/sft_train.py \
    --model $BASE_MODEL \
    --data data/preferences/preferences.jsonl \
    --out training/outputs/sft \
    --epochs 2

## 5. Phase 6 — DPO (starts from the SFT adapter)

In [ ]:
!python training/dpo_train.py \
    --base $BASE_MODEL \
    --sft_adapter training/outputs/sft \
    --out training/outputs/dpo \
    --data data/preferences/preferences.jsonl \
    --epochs 1 --beta 0.1

## 6. Phase 7a — Predictions on the held-out set

In [ ]:
!python evaluation/infer.py --base $BASE_MODEL --tag base
!python evaluation/infer.py --base $BASE_MODEL --adapter training/outputs/sft --tag sft
!python evaluation/infer.py --base $BASE_MODEL --adapter training/outputs/dpo --tag dpo

In [ ]:
import json
b = [json.loads(l) for l in open('results/preds_base.jsonl')]
d = [json.loads(l) for l in open('results/preds_dpo.jsonl')]
print('PROMPT:\n', b[0]['prompt'][:300])
print('\n=== BASE ===\n', b[0]['output'][:500])
print('\n=== DPO ===\n', d[0]['output'][:500])

## 7. Phase 7b — Judge the outputs (needs GEMINI_API_KEY)

`google-genai` installs cleanly in the Studio Lab env. Writes `results/eval_report.md`.

In [ ]:
!pip install -q google-genai python-dotenv
import getpass, os
os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

In [ ]:
# DPO vs base — the headline number
!python evaluation/run_eval.py --a results/preds_base.jsonl --b results/preds_dpo.jsonl

In [ ]:
# DPO vs SFT — isolates the lift that comes specifically from preference optimization
!python evaluation/run_eval.py --a results/preds_sft.jsonl --b results/preds_dpo.jsonl

In [ ]:
from IPython.display import Markdown
Markdown(open('results/eval_report.md').read())

## 8. Save your work

No `google.colab.files` here — everything is already written to the **persistent project dir**.
To pull artifacts to your laptop, use the JupyterLab **file browser** on the left:
right-click `results/eval_report.md` (or `training/outputs/`) → **Download**.

Or commit them straight back to the repo:

In [ ]:
# !git config user.email 'you@example.com'
# !git config user.name 'Your Name'
# !git add results/eval_report.md results/eval_report.json
# !git commit -m 'Add eval results'
# !git push  # needs a GitHub token configured

# Zip the adapters for an easy single download from the file browser:
!cd training/outputs && zip -qr ../../results/adapters.zip sft dpo && echo 'wrote results/adapters.zip'